# 1. Project Introduction

Welcome! In this notebook, we will explore **AdaBoost** (Adaptive Boosting), one of the earliest and most popular boosting algorithms.

### What is AdaBoost?
* It is a **supervised learning** classifier.
* Boosting is a sequential ensemble method. Rather than training trees independently (like Random Forest), AdaBoost trains them **one after another**.
* Each successive tree (often a very simple tree with a depth of 1, called a **decision stump**) focuses on correcting the errors made by the previous trees.
* **Adaptive**: It does this by increasing the weights of misclassified data points, so the next stump pays more attention to hard cases.

### Why does it exist?
* It turns weak learners (models that perform just slightly better than random guessing) into a strong collective ensemble.

### Real-World Use Cases:
* **Facial Detection**: Historically used in early face-detection software (e.g., Viola-Jones algorithm).
* **Patient Churn**: Predicting subscriber drop-off.


# 2. Problem Statement

* **Goal**: Predict if a credit card user will **Malignant (1)** on their next payment or pay **On Time (0)**.
* **Business Value**: Minimizes banking losses from credit malignants.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn import metrics


# 4. Create Synthetic Dataset

We define features for **100 credit card accounts**.
* **Mean Radius**: Account holder's age.
* **Mean_Texture**: Monthly income in k$.
* **Mean_Perimeter**: Number of billing cycles missed in the past year (0 to 6).
* **Tumor_Type**: Target classification label.


In [ ]:
# Load scikit-learn breast cancer dataset
from sklearn.datasets import load_breast_cancer
import pandas as pd
cancer = load_breast_cancer(as_frame=True)
raw_df = cancer.frame

df = pd.DataFrame({
    'Mean_Radius': raw_df['mean radius'],
    'Mean_Texture': raw_df['mean texture'],
    'Mean_Perimeter': raw_df['mean perimeter'],
    'Tumor_Type': raw_df['target']
})

print("Shape:", df.shape)
print(df.head())


# 5. Exploratory Data Analysis (EDA)


In [ ]:
# Chart 1: Distribution of Missed Payments by Malignant Outcome
plt.figure(figsize=(8, 4))
sns.boxplot(x='Tumor_Type', y='Mean_Radius', data=df, palette='Set2')
plt.title('Mean Radius vs. Tumor Type')
plt.xlabel('Tumor Type (0 = Malignant, 1 = Benign)')
plt.ylabel('Mean Radius')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* Malignanters (class 1) have a median of 2-3 mean perimeter, while non-malignanters have 0 or 1.


In [ ]:
# Cleaning check
print("Null count:", df.isnull().sum().sum())


In [ ]:
# Feature Selection
X = df[['Mean_Radius', 'Mean_Texture', 'Mean_Perimeter']]
y = df['Tumor_Type']


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 9. Model Building

* **How it works**: AdaBoost trains a sequence of decision stumps (one-split trees). After each stump is built, the weights of incorrectly classified training records are increased. The final prediction is a weighted sum of predictions from all stumps.


In [ ]:
# Initialize AdaBoost classifier with 30 stumps
model = AdaBoostClassifier(n_estimators=30, random_state=42)


In [ ]:
# Train AdaBoost
model.fit(X_train, y_train)


In [ ]:
# Predict labels
predictions = model.predict(X_test)


In [ ]:
# Compute metrics
accuracy = metrics.accuracy_score(y_test, predictions)
precision = metrics.precision_score(y_test, predictions)
recall = metrics.recall_score(y_test, predictions)
f1 = metrics.f1_score(y_test, predictions)
conf_matrix = metrics.confusion_matrix(y_test, predictions)

# Print metrics in plain English
print(f"Accuracy Score: {accuracy:.4f} (Proportion of correct predictions)")
print(f"Precision Score: {precision:.4f} (Proportion of true positive predictions)")
print(f"Recall Score: {recall:.4f} (Proportion of actual positives caught)")
print(f"F1 Score: {f1:.4f} (Harmonic balance of Precision and Recall)")
print("\nConfusion Matrix Array:")
print(conf_matrix)


### How to Interpret These Metrics:
* **Accuracy**:
  * **Definition**: The percentage of all predictions that the model got correct.
  * **Interpretation**: An accuracy of around **90% to 95%** means the model correctly diagnoses the tumor type for 90-95% of patients in our test set.
* **Precision**:
  * **Definition**: Out of all tumors predicted as positive (Benign), what percentage were actually benign?
  * **Interpretation**: A high precision (e.g., **94%**) means that when the model predicts a tumor is benign, it is correct 94% of the time, resulting in very few false alarms.
* **Recall (Sensitivity)**:
  * **Definition**: Out of all actual positive cases (Benign), what percentage did we successfully identify?
  * **Interpretation**: A recall of around **93%** means the model successfully identified 93% of all benign tumors.
* **F1-Score**:
  * **Definition**: The harmonic mean of precision and recall. It balances both metrics to give a single overall performance score.
* **Confusion Matrix**:
  * **True Negatives (TN)**: Malignant tumors correctly classified as Malignant.
  * **False Positives (FP)**: Malignant tumors incorrectly classified as Benign (this is a critical medical issue because a patient with cancer is told they are healthy!).
  * **False Negatives (FN)**: Benign tumors incorrectly classified as Malignant.
  * **True Positives (TP)**: Benign tumors correctly classified as Benign.


In [ ]:
# Plot 1: Confusion Matrix Heatmap
conf_matrix = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Predicted Benign', 'Predicted Malignant'], 
            yticklabels=['Actual Benign', 'Actual Malignant'])
plt.title('AdaBoost Confusion Matrix')
plt.show()


In [ ]:
# Plot 2: Feature Importances
plt.figure(figsize=(6, 4))
sns.barplot(x=model.feature_importances_, y=X.columns, palette='copper')
plt.title('AdaBoost Feature Importances')
plt.xlabel('Importance score')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* `Mean_Perimeter` is identified as the most important feature.


# 14. Model Interpretation

* **Sequential Learning**: AdaBoost builds successive stumps where each stump's voting power depends on its error rate.
* **Feature Importance**: Stumps split on features that clear up prediction errors for weighted records.


# 15. Conclusion
* AdaBoost builds predictive capability iteratively.


# 16. Beginner ML Dictionary

Here are simple, one-sentence explanations of common Machine Learning terms to help you review:

* **Feature**: An input variable or column in your dataset used to make predictions (e.g., hours studied).
* **Target**: The output variable or label you want the model to predict (e.g., final exam score).
* **Training Data**: The portion of the dataset used to teach the model and find patterns.
* **Testing Data**: The portion of the dataset held back to evaluate how well the model performs on new, unseen data.
* **Prediction**: The output value generated by the trained model when given new input features.
* **Overfitting**: A scenario where the model learns the training data too well, including its noise, and performs poorly on new data.
* **Underfitting**: A scenario where the model is too simple to learn the underlying patterns in the training data, leading to poor performance on both training and test data.
* **Model**: The mathematical representation of the patterns learned from the training data by the algorithm.
* **Algorithm**: The set of rules or mathematical procedures followed to build the model from the data (e.g., Linear Regression).
* **Accuracy**: The percentage of correct predictions made by a classification model.
* **Cluster**: A group of similar data points grouped together by an unsupervised learning algorithm based on their characteristics.
* **Centroid**: The center point of a cluster, representing the average location of all data points belonging to that cluster.
